### Importing Libraries

In [ ]:
import torch # deep learning framework
import torch.nn as nn # neural network module
import torch.optim as optim # optimization algorithms

from torchvision import datasets, transforms # datasets and data transformations
from torch.utils.data import DataLoader # data loading utility
from peft import LoraConfig, get_peft_model # used for parameter-efficient fine-tuning (PEFT)
import timm # contain pretrained image models 

from tqdm import tqdm # progress bar library

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device) # print the device being used (GPU or CPU)

cuda


### CIFAR-10 Dataset

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)), # resize CIFAR-10 images from 32x32x3 to 224x224x3
    transforms.ToTensor(), # convert images to tensors((3x224x224) because pytorch tensor takes(C, H, W)) and scale pixel values to [0, 1]
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5], # mean for each color channel (R, G, B)
        std=[0.5, 0.5, 0.5] # standard deviation for each color channel (R, G, B)
    )  # normalize images
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform) # load CIFAR-10 training dataset
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform) # load CIFAR-10 test dataset

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) # create data loader for training dataset
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False) # create data loader for test dataset

### Training Function (Standard PyTorch training loop)

In [4]:
def train_model(model, epochs=10):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(model.parameters(), lr=1e-3) # AdamW optimizer with learning rate of 0.001
    #model.parameters() returns all the trainable parameters of the model, which are updated during training to minimize the loss function 
    losses = [] # list to store average loss for each epoch, which can be used for plotting the training loss curve later

    for epoch in range(epochs):

        model.train() # set the model to training mode, which enables features like dropout and batch normalization to work correctly during training

        running_loss = 0 # variable to accumulate the total loss for the current epoch, which will be used to calculate the average loss at the end of the epoch

        for images, labels in tqdm(train_loader):

            images = images.to(device) # images.shape = (64, 3, 224, 224) move the batch of images to the specified device GPU 
            labels = labels.to(device) # labels.shape = (64,) move the batch of labels to the specified device GPU

            optimizer.zero_grad() # clear the gradients of all optimized parameters, which is necessary before performing backpropagation to prevent accumulation of gradients from previous iterations

            # Forward pass 
            outputs = model(images) # forward pass: compute the model's predictions for the input images, which will be used to calculate the loss against the true labels
            # outputs.shape = (64, 10) because we have 10 classes in CIFAR-10, so the model outputs a probability distribution over the 10 classes for each image in the batch

            # Loss calculation
            loss = criterion(outputs, labels) # compute the loss by comparing the model's predictions (outputs) with the true labels, which quantifies how well the model is performing
            # loss is a scalar value that represents the average loss for the batch, which will be used to update the model's parameters during backpropagation
            
            # Backward pass 
            loss.backward() # backpropagation: compute the gradients of the loss with respect to the model's parameters, which will be used by the optimizer to update the parameters in the direction that minimizes the loss
            
            # Update parameters
            optimizer.step() # update the model's parameters based on the computed gradients, which is a crucial step in the training process to improve the model's performance

            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader) # calculate the average loss for the epoch by dividing the total accumulated loss (running_loss) by the number of batches (len(train_loader)), which gives a measure of how well the model is learning over the course of the epoch

        losses.append(avg_loss)

        print(f"Epoch {epoch+1}: {avg_loss:.4f}")

    return losses

### Evaluation

In [5]:
def evaluate(model):

    model.eval() # set the model to evaluation mode, which disables features like dropout and batch normalization that are only used during training, ensuring that the model's behavior is consistent during evaluation

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = outputs.argmax(1) # get the predicted class labels by finding the index of the maximum value in the output tensor along dimension 1 (which corresponds to the class probabilities), resulting in a tensor of predicted class indices for each image in the batch

            correct += (preds == labels).sum().item() # count the number of correct predictions by comparing the predicted labels (preds) with the true labels (labels), summing the number of matches, and converting it to a Python scalar using .item()

            total += labels.size(0) # batch size, which is the number of samples in the current batch, and add it to the total count of samples processed so far

    acc = 100 * correct / total

    return acc

In [6]:
cnn_model=timm.create_model('resnet18', pretrained=True, num_classes=10).to(device) # create a ResNet-18 model pre-trained on ImageNet, with 10 output classes for CIFAR-10
print(cnn_model) # print the model architecture
# timm.create_model() provides 100+pretrained models. 
# pretrained=True loads weights that have been pre-trained on the ImageNet dataset, which can help improve performance and speed up convergence when fine-tuning the model on a new task like CIFAR-10.
# ImageNet is a large dataset of images with 1000 classes, and using a model pre-trained on ImageNet allows the model to leverage learned features that can be beneficial for the CIFAR-10 classification task, even though CIFAR-10 has only 10 classes.
# instead of random weightd initialization, using a pre-trained model can lead to faster convergence and better performance, especially when the new dataset is small or similar to the original dataset used for pre-training.
# this is called transfer learning, where we transfer the knowledge learned from one task (ImageNet classification) to another related task (CIFAR-10 classification) by fine-tuning the pre-trained model on the new dataset.

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (b

In [7]:
mae_backbone = timm.create_model("vit_base_patch16_224",pretrained=False,num_classes=0).to(device) # create a ViT base model without pre-training and no output classes, to be used as a backbone for MAE
print(mae_backbone) # print the MAE backbone architecture

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropou

In [8]:
ijepa_backbone = timm.create_model("vit_small_patch16_224", pretrained=False, num_classes=0).to(device) # create a ViT small model without pre-training and no output classes, to be used as a backbone for IJEPAs
print(ijepa_backbone) # print the IJEPAs backbone architecture

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True, bias=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropou

In [ ]:
class LinearProbe(nn.Module):

    def __init__(self,backbone,feature_dim):

        super().__init__()

        self.backbone = backbone

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.fc = nn.Linear(feature_dim,10)

    def forward(self, x):

        features = self.backbone(x)
        return self.fc(features)

In [ ]:
class LoRAClassifier(nn.Module):

    def __init__(self,backbone,feature_dim):

        super().__init__()

        self.backbone = backbone
        self.fc = nn.Linear(feature_dim,10)

    def forward(self, x):

        features = self.backbone(x)
        return self.fc(features)


In [11]:
print("\nTraining CNN")
train_model(cnn_model, epochs=10)
cnn_acc = evaluate(cnn_model)
print("CNN Accuracy =", cnn_acc)


Training CNN


100%|██████████| 782/782 [06:27<00:00,  2.02it/s]


Epoch 1: 0.4155


100%|██████████| 782/782 [05:35<00:00,  2.33it/s]


Epoch 2: 0.1656


...

Epoch 8: 0.0369


100%|██████████| 782/782 [04:28<00:00,  2.91it/s]


Epoch 9: 0.0333


100%|██████████| 782/782 [04:20<00:00,  3.01it/s]


Epoch 10: 0.0394
CNN Accuracy = 93.38


In [ ]:
mae_probe = LinearProbe(mae_backbone,feature_dim=768).to(device) # create a linear probe for the MAE backbone, with feature dimension 768 (ViT base output dimension)

print("\nTraining MAE Probe")

train_model(mae_probe, epochs=10)

mae_probe_acc = evaluate(mae_probe)


Training MAE Probe


100%|██████████| 782/782 [12:31<00:00,  1.04it/s]


Epoch 1: 1.9162


100%|██████████| 782/782 [17:49<00:00,  1.37s/it]


Epoch 2: 1.8090


...

Epoch 8: 1.6766


100%|██████████| 782/782 [18:06<00:00,  1.39s/it]


Epoch 9: 1.6731


100%|██████████| 782/782 [18:46<00:00,  1.44s/it] 


Epoch 10: 1.6651


In [ ]:
mae_lora_cfg = LoraConfig(r=8,lora_alpha=16,target_modules=["qkv"])

mae_lora = get_peft_model(mae_backbone,mae_lora_cfg)

mae_lora_model = LoRAClassifier(mae_lora,feature_dim=768).to(device) # create a LoRA classifier for the MAE backbone, with feature dimension 768 (ViT base output dimension)

print("\nTraining MAE LoRA")
train_model(mae_lora_model, epochs=10)
mae_lora_acc = evaluate(mae_lora_model)


Training MAE LoRA


100%|██████████| 782/782 [1:26:19<00:00,  6.62s/it]


Epoch 1: 1.6984


100%|██████████| 782/782 [1:25:02<00:00,  6.53s/it]


Epoch 2: 1.4560


...

Epoch 8: 1.0981


100%|██████████| 782/782 [1:11:04<00:00,  5.45s/it]


Epoch 9: 1.0701


100%|██████████| 782/782 [1:10:58<00:00,  5.45s/it]


Epoch 10: 1.0497


In [ ]:
ijepa_probe = LinearProbe(ijepa_backbone,feature_dim=384).to(device) # create a linear probe for the IJEPAs backbone, with feature dimension 384 (ViT small output dimension)
print("\nTraining I-JEPA Probe")
train_model(ijepa_probe, epochs=10)
ijepa_probe_acc = evaluate(ijepa_probe)



Training I-JEPA Probe


100%|██████████| 782/782 [03:10<00:00,  4.10it/s]


Epoch 1: 1.9412


100%|██████████| 782/782 [03:10<00:00,  4.11it/s]


Epoch 2: 1.8686


...

Epoch 8: 1.7668


100%|██████████| 782/782 [03:10<00:00,  4.10it/s]


Epoch 9: 1.7576


100%|██████████| 782/782 [03:10<00:00,  4.10it/s]


Epoch 10: 1.7545


In [ ]:
ijepa_lora_cfg = LoraConfig(r=8,lora_alpha=16,target_modules=["qkv"])

ijepa_lora = get_peft_model(ijepa_backbone,ijepa_lora_cfg)

ijepa_lora_model = LoRAClassifier(ijepa_lora,feature_dim=384).to(device) # create a LoRA classifier for the IJEPAs backbone, with feature dimension 384 (ViT small output dimension)
print("\nTraining I-JEPA LoRA")
train_model(ijepa_lora_model,epochs=10)

ijepa_lora_acc = evaluate(ijepa_lora_model)


Training I-JEPA LoRA


100%|██████████| 782/782 [21:17<00:00,  1.63s/it]


Epoch 1: 1.7164


100%|██████████| 782/782 [22:41<00:00,  1.74s/it]


Epoch 2: 1.4475


...

Epoch 8: 1.1004


100%|██████████| 782/782 [21:37<00:00,  1.66s/it]


Epoch 9: 1.0783


100%|██████████| 782/782 [23:06<00:00,  1.77s/it]


Epoch 10: 1.0508


In [ ]:
print("\nRESULTS")
print(f"CNN Accuracy: {cnn_acc:.2f}%")
print(f"MAE Probe Accuracy: "f"{mae_probe_acc:.2f}%")
print(f"MAE LoRA Accuracy: "f"{mae_lora_acc:.2f}%")
print(f"I-JEPA Probe Accuracy: "f"{ijepa_probe_acc:.2f}%")
print(f"I-JEPA LoRA Accuracy: "f"{ijepa_lora_acc:.2f}%")


RESULTS
CNN Accuracy: 93.38%
MAE Probe Accuracy: 37.84%
MAE LoRA Accuracy: 58.02%
I-JEPA Probe Accuracy: 36.83%
I-JEPA LoRA Accuracy: 56.88%
